# Lab 07

## Zadanie 1

Przygotuj demo programu znajdującego miejsca zerowe metodą Newtona. Wykorzystując `argparse` ([link](https://docs.python.org/3.6/library/argparse.html#module-argparse)) lub `optparse` ([link](https://docs.python.org/3.6/library/optparse.html)) obsłuż:
- ustalane punktu startowego,
- wielkość kroku w pochodnej,
- ilość kroków metody,
- dokładność
- pomoc

Program uruchamiamy podając, np.:

`./newton.py x**2+x+1 -h 0.00001`

Zadanie gotowe do wglądu w osobnym pliku newton.py

## Zadanie 2

Zadanie należy wykonać wykorzystując program `BagOfWords` z poprzednich zajęć.


Przerób ją tak, żeby interpunkcja, cyfry i wszelkie inne znaki nie przeszkadzały w parsowaniu tekstu. Uruchom ja na [tekście hamleta](http://www.gutenberg.org/cache/epub/1787/pg1787.txt). Ile razy występuje słowo hamlet? Jak brzmi 10 najczęściej występujących słów?

In [3]:
import re
from collections import Counter
from urllib.request import urlopen

# Cyfry, podkreślniki i interpunkcja nie wchodzą do słów
# Pierwszy człon regexu [^\W\d_]+ - dowolny znak, który nie jest \W (nie znak słowa), liczbą (\d) ani _, plus(+) mówi że mamy jedną lub więcej takich liter pod rząd
# Drugi człon regexu (?:['’][^\W\d_]+)* - [''] oznaczają dosłownie te dwa apostrofy, [^\W\d_]+ to jedna lub więcej liter. Cały człon to apostrof + litery, ?: ... * mówi że może wystąpić 0 lub więcej razy
WORD_RE = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*")       


class BagOfWords(object):
    def __init__(self, source=""):
        self._counts = Counter()

        text = source.read() if hasattr(source, "read") else source
        self._counts.update(self._tokenize(text))                            # Nie używamy splita, żeby znaki inne niż litery nie rozróżniały słowa od innych

    # casefold() łączy "Hamlet", "HAMLET" i "hamlet" w jedno słowo
    # @staticmethod aby _tokenize() była zwykłą funkcją, a nie częścią klasy z self
    @staticmethod
    def _tokenize(text):
        return WORD_RE.findall(text.casefold())

    def _sorted_words(self):
        # Od najczęstszego do najrzadszego
        return sorted(self._counts, key=lambda w: (-self._counts[w], w))

    def __str__(self):
        return ", ".join("%s:%d" % (w, self._counts[w]) for w in self._sorted_words())

    __repr__ = __str__

    def __contains__(self, word):
        return word in self._counts

    def __iter__(self):
        return iter(self._sorted_words())

    def __len__(self):
        return len(self._counts)

    def __add__(self, other):
        if not isinstance(other, BagOfWords):
            return NotImplemented
        result = BagOfWords()
        result._counts = self._counts + other._counts  # Nowy Counter
        return result

    def __getitem__(self, word):
        return self._counts[word]  # Brakujące słowo da 0

    def __setitem__(self, word, count):
        if count > 0:
            self._counts[word] = count
        else:
            self._counts.pop(word, None)


if __name__ == "__main__":
    url = "https://standardebooks.org/ebooks/william-shakespeare/hamlet/text/single-page"                            # Alternatywa do Project Gutenberg, który jest obecnie niedostępny
    raw = urlopen(url).read().decode("utf-8-sig")                                                                    # Kod zwraca .html a nie .txt, stąd na tej stronie klasa nie będzie działać idealnie



    bag = BagOfWords(raw)
    print("hamlet:", bag["hamlet"])
    print("Top 10:", list(bag)[:10])
    print([(w, bag[w]) for w in list(bag)[:10]])

hamlet: 481
Top 10: ['span', 'td', 'c', 'tr', 'z', 'br', 'epub', 'type', 'persona', 'the']
[('span', 5358), ('td', 4837), ('c', 3061), ('tr', 2460), ('z', 2272), ('br', 2267), ('epub', 2187), ('type', 2185), ('persona', 1366), ('the', 1231)]


In [1]:

# Oryginał do porównania
from collections import Counter


class BagOfWords(object):
    def __init__(self, source=""):
        self._counts = Counter()

        # Plik ma metodę read(), a string nie
        text = source.read() if hasattr(source, "read") else source
        self._counts.update(text.split())

    def _sorted_words(self):
        # Od najczęstszego do najrzadszego
        return sorted(self._counts, key=lambda w: (-self._counts[w], w))

    def __str__(self):
        return ", ".join("%s:%d" % (w, self._counts[w]) for w in self._sorted_words())

    __repr__ = __str__

    def __contains__(self, word):
        return word in self._counts

    def __iter__(self):
        return iter(self._sorted_words())

    def __len__(self):
        return len(self._counts)

    def __add__(self, other):
        if not isinstance(other, BagOfWords):
            return NotImplemented
        result = BagOfWords()
        result._counts = self._counts + other._counts  # Nowy Counter
        return result

    def __getitem__(self, word):
        return self._counts[word]  # Brakujące słowo da 0

    def __setitem__(self, word, count):
        if count > 0:
            self._counts[word] = count
        else:
            self._counts.pop(word, None)  # Słowo z licznikiem 0 nie należy do worka

## Zadanie 3

Wykorzystując `pickle` zapisz i odczytaj klasy z poprzedniego zadania nakarmonej Hamletem. Porównaj metody i rozmiar.

In [ ]:
import gzip
import os
import pickle
import re
from urllib.request import urlopen


def size(path):
    return os.path.getsize(path)


if __name__ == "__main__":
    url = "https://standardebooks.org/ebooks/william-shakespeare/hamlet/text/single-page"
    raw = urlopen(url).read().decode("utf-8-sig")
    text = re.sub(r"<[^>]+>", " ", raw)          # Wycięcie tagów HTML

    bag = BagOfWords(text)
    print("Różnych słów:", len(bag))

    for proto in range(pickle.HIGHEST_PROTOCOL + 1):
        fname = f"hamlet_p{proto}.pkl"
        with open(fname, "wb") as f:
            pickle.dump(bag, f, protocol=proto)                                    # Zapis do pliku
        print(f"protokół {proto}: {size(fname):>8} B")

    with open(f"hamlet_p{pickle.HIGHEST_PROTOCOL}.pkl", "rb") as f:                # Odczyt i porównywanie obiektu
        loaded = pickle.load(f)

    assert len(loaded) == len(bag)
    assert str(loaded) == str(bag)
    assert loaded["hamlet"] == bag["hamlet"]
    assert loaded is not bag

    blob = pickle.dumps(bag, protocol=pickle.HIGHEST_PROTOCOL)                     # Zapis i odczyt w byte
    print("dumps():", len(blob), "B")
    assert str(pickle.loads(blob)) == str(bag)

    with gzip.open("hamlet.pkl.gz", "wb") as f:
        pickle.dump(bag, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("pickle+gzip:", size("hamlet.pkl.gz"), "B")

    with gzip.open("hamlet.pkl.gz", "rb") as f:
        assert str(pickle.load(f)) == str(bag)

    with open("hamlet_dict.pkl", "wb") as f:
        pickle.dump(dict(bag._counts), f, protocol=pickle.HIGHEST_PROTOCOL)        # Wyciąga sam słownik
    print("sam dict:", size("hamlet_dict.pkl"), "B")

    with open("hamlet.txt", "w", encoding="utf-8") as f:                          
        f.write(str(bag))
    print("tekst (__str__):", size("hamlet.txt"), "B")

Różnych słów: 5034
protokół 0:    93695 B
protokół 1:    92630 B
protokół 2:    92569 B
protokół 3:    92569 B
protokół 4:    58081 B
protokół 5:    58081 B
dumps(): 58081 B
pickle+gzip: 25402 B
sam dict: 58008 B
tekst (__str__): 53431 B


## Zadanie 4

Wykorzystując [https://gist.github.com/pamelafox/986163](https://gist.github.com/pamelafox/986163) podaj aktualną godzinę we:
- wszystkich krajach, wyświetlając je zgrupowane względem kontynentów,
- przeprowadź symulacyjne wyświetlanie kolejnych krajów w miarę jak w danym kraju wybija północ, opóżnienie wyświetlania ustaw proporcjonalne do realnego czasu

In [112]:
import pytz
from datetime import datetime
import countryinfo as inf


def timezones():
    inf.countries.sort(key = lambda country: country['continent'])

    for c in inf.countries:
        time_info = datetime.now(pytz.timezone(c['timezones'][0]))
        print(f"{c['name']}: {time_info.hour}:{time_info.minute}:{time_info.second}")

timezones()

Mauritius: 17:42:41
Seychelles: 17:42:41
Djibouti: 16:42:41
Eritrea: 16:42:41
Ethiopia: 16:42:41
Kenya: 16:42:41
Madagascar: 16:42:41
Somalia: 16:42:41
Tanzania: 16:42:41
Uganda: 16:42:41
Comoros: 16:42:41
Burundi: 15:42:41
Botswana: 15:42:41
Egypt: 15:42:41
Lesotho: 15:42:41
Libya: 15:42:41
Malawi: 15:42:41
Mozambique: 15:42:41
Namibia: 15:42:41
Rwanda: 15:42:41
Sudan: 15:42:41
Zambia: 15:42:41
Zimbabwe: 15:42:41
South Africa: 15:42:41
Swaziland: 15:42:41
Angola: 14:42:41
Benin: 14:42:41
Democratic Republic of the Congo: 14:42:41
Republic of the Congo: 14:42:41
Cameroon: 14:42:41
Gabon: 14:42:41
Niger: 14:42:41
Nigeria: 14:42:41
Tunisia: 14:42:41
Algeria: 14:42:41
Central African Republic: 14:42:41
Chad: 14:42:41
Equatorial Guinea: 14:42:41
Morocco: 14:42:41
Burkina Faso: 13:42:41
CÃ´te d'Ivoire: 13:42:41
Ghana: 13:42:41
The Gambia: 13:42:41
Guinea: 13:42:41
Guinea-Bissau: 13:42:41
Liberia: 13:42:41
Mali: 13:42:41
Mauritania: 13:42:41
Sierra Leone: 13:42:41
Senegal: 13:42:41
SÃ£o TomÃ

In [114]:
import time
 

def seconds_to_midnight(country, now_utc):
    tz = pytz.timezone(country['timezones'][0])
    local = now_utc.astimezone(tz)
    since_midnight = local.hour * 3600 + local.minute * 60 + local.second
    return (86400 - since_midnight) % 86400


def countries_to_midnight_order(now_utc):
    schedule = [(seconds_to_midnight(c, now_utc), c) for c in inf.countries]
    schedule.sort(key=lambda touple: touple[0])
    return schedule


def simulate_midnight(SIM_SPEED):
    now_utc = datetime.now(pytz.utc)  
    schedule = countries_to_midnight_order(now_utc)
    previous = schedule[0][0]

    for t, country in schedule:
        time.sleep((t - previous) / SIM_SPEED)  
        local = now_utc.astimezone(pytz.timezone(country['timezones'][0]))
        print(f"{country['name']:<35} "
              f"({t // 3600:02d}h {t % 3600 // 60:02d}m od startu symulacji, "
              f"teraz jest: {local.strftime('%H:%M')})")
        previous = t


if __name__ == "__main__":
    simulate_midnight(SIM_SPEED = 1400)

Papua New Guinea                    (00h 11m od startu symulacji, teraz jest: 23:48)
Federated States of Micronesia      (00h 11m od startu symulacji, teraz jest: 23:48)
Japan                               (01h 11m od startu symulacji, teraz jest: 22:48)
North Korea                         (01h 11m od startu symulacji, teraz jest: 22:48)
South Korea                         (01h 11m od startu symulacji, teraz jest: 22:48)
East Timor                          (01h 11m od startu symulacji, teraz jest: 22:48)
Palau                               (01h 11m od startu symulacji, teraz jest: 22:48)
Brunei Darussalam                   (02h 11m od startu symulacji, teraz jest: 21:48)
People's Republic of China          (02h 11m od startu symulacji, teraz jest: 21:48)
Mongolia                            (02h 11m od startu symulacji, teraz jest: 21:48)
Malaysia                            (02h 11m od startu symulacji, teraz jest: 21:48)
Philippines                         (02h 11m od startu symulacji,

## Zadanie 5

Dla klasy `BagOfWords` napisz metody `save` oraz `load` wykorzystujące `json`'a do zapisu i odczytu danych.

In [125]:
import json
from collections import Counter

class BagOfWords(object):
    def __init__(self, source=""):
        self._counts = Counter()

        # Plik ma metodę read(), a string nie
        text = source.read() if hasattr(source, "read") else source
        self._counts.update(text.split())

    def _sorted_words(self):
        # Od najczęstszego do najrzadszego
        return sorted(self._counts, key=lambda w: (-self._counts[w], w))

    def __str__(self):
        return ", ".join("%s:%d" % (w, self._counts[w]) for w in self._sorted_words())

    __repr__ = __str__

    def __contains__(self, word):
        return word in self._counts

    def __iter__(self):
        return iter(self._sorted_words())

    def __len__(self):
        return len(self._counts)

    def __add__(self, other):
        if not isinstance(other, BagOfWords):
            return NotImplemented
        result = BagOfWords()
        result._counts = self._counts + other._counts  # Nowy Counter
        return result

    def __getitem__(self, word):
        return self._counts[word]  # Brakujące słowo da 0

    def __setitem__(self, word, count):
        if count > 0:
            self._counts[word] = count
        else:
            self._counts.pop(word, None)  # Słowo z licznikiem 0 nie należy do worka






    def save(self,target):
        if hasattr(target, "write"):                                                   # Sprawdza czy podano ścieżkę, czy plik
            json.dump(dict(self._counts), target, ensure_ascii=False, indent=2)           
        else:
            with open(target, "w", encoding="utf-8") as f:
                json.dump(dict(self._counts), f, ensure_ascii=False, indent=2)         # ensure_ascii oraz encoding="utf-8" aby polskie znaki były zapisywane czytelnie


    def load(source):
        if hasattr(source, "read"):
            data = json.load(source)
        else:
            with open(source, "r", encoding="utf-8") as f:
                data = json.load(f)


        for word, count in data.items():
            bag[word] = count
        return bag


bag = BagOfWords("ala ma kota ala ma psa")
bag.save("bag.json")

print(BagOfWords.load("bag.json"))


ala:2, ma:2, kota:1, psa:1
